In [30]:
# Noun-only Didakta pipeline: trains on nouns using didakta_1 (cases) and saves errors/context under ex1
import os
import re
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# --- portable paths: repo root found via .git, so this runs on any machine ---
from pathlib import Path
def _find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists():
            return cand
    return p
REPO = _find_repo()
EXP_DIR = REPO / "experiments" / "ex1-noun-tagging"
DATA = REPO / "data"
RESULTS = EXP_DIR / "results"
MODELS = EXP_DIR / "models"
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
train_candidates = [
    "Odyssey5- Didakta_no_unmatched.csv",
    "Iliad1- Didakta_no_unmatched.csv",
]
train_paths = [os.path.join(DATA, name) for name in train_candidates if os.path.exists(os.path.join(DATA, name))]
if not train_paths:
    raise FileNotFoundError("Could not find Odyssey/Iliad training CSV files.")

# helper to normalize tag text
def normalize_tag_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace(".", "")
    text = re.sub(r"\d+$", "", text)
    return text.strip()

# helper to build feature text from a row
def row_to_text(row, columns):
    parts = []
    for column in columns:
        value = row.get(column, "")
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text:
            parts.append(f"{column}={text}")
    return " | ".join(parts)

# load tag list to identify case tags, articles, pronouns
tag_list_path = os.path.join(DATA, "didakta-tag-list.csv")
tag_list_df = pd.read_csv(tag_list_path, dtype=str)
case_tags = set(tag_list_df.loc[~tag_list_df["is_category"].astype(str).str.lower().eq("true"), "didakta_tag"].dropna().astype(str))
article_tags = set(tag_list_df.loc[tag_list_df["category"].eq("Article"), "didakta_tag"].dropna().astype(str))
pronoun_tags = set(tag_list_df.loc[tag_list_df["category"].astype(str).str.contains("pronoun", case=False, na=False), "didakta_tag"].dropna().astype(str))

# load training data
train_df = pd.concat([pd.read_csv(path, dtype=str) for path in train_paths], ignore_index=True)
for col in ["didakta_1", "didakta_2", "didakta_3"]:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna("").astype(str).map(normalize_tag_text)
    else:
        train_df[col] = ""

# determine POS column
if "pos" not in train_df.columns and "postag" not in train_df.columns:
    raise ValueError("No POS column found in the training data.")
pos_col = "pos" if "pos" in train_df.columns else "postag"

# select noun rows only, require didakta_1 to be a case tag, and exclude articles/pronouns
noun_mask = train_df[pos_col].fillna("").astype(str).str.lower().str.startswith("n")
article_mask = train_df["didakta_1"].isin(article_tags)
pronoun_mask = train_df["didakta_1"].isin(pronoun_tags)
case_mask = train_df["didakta_1"].isin(case_tags)

noun_df = train_df[noun_mask & case_mask & ~article_mask & ~pronoun_mask].copy()

print(f"Noun rows kept: {len(noun_df)}")
if len(noun_df) == 0:
    raise ValueError("No noun rows with case labels found after filtering.")

# choose feature columns (exclude didakta columns)
exclude_cols = {"didakta", "didakta_1", "didakta_2", "didakta_3", "didakta_4"}
feature_cols = [c for c in noun_df.columns if c not in exclude_cols and c not in ["primary_annotators", "secondary_annotators"]]
if not feature_cols:
    feature_cols = [c for c in ["ref", "greek", "lemma", pos_col, "rel", "syn", "line"] if c in noun_df.columns]

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

# build X and y (pandas Series to preserve indices)
X = noun_df.apply(lambda row: row_to_text(row, feature_cols), axis=1).astype(str)
y = noun_df["didakta_1"].astype(str)

# restrict to most frequent labels to avoid extreme sparsity
max_classes = 40
label_counts = Counter(y)
if len(label_counts) > max_classes:
    keep_labels = {label for label, _ in label_counts.most_common(max_classes)}
    keep_mask = y.isin(keep_labels)
    X = X[keep_mask]
    y = y[keep_mask]
    noun_df = noun_df[keep_mask].copy()
    print(f"Restricted noun task to top {max_classes} labels.")

print(f"Noun-task rows: {len(y)}")
print(f"Noun-task unique labels: {len(set(y))}")

# split while preserving original indices so we can map errors back to noun_df
try:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------- Optional transformer-based features (Sentence-BERT) ----------
transformer_available = False
transformer_model = None
try:
    from sentence_transformers import SentenceTransformer
    # try compact SBERT models (may download if not present)
    for cand in ['sentence-transformers/all-mpnet-base-v2', 'sentence-transformers/paraphrase-mpnet-base-v2']:
        try:
            transformer_model = SentenceTransformer(cand)
            transformer_available = True
            print(f'Loaded transformer embedder: {cand}')
            break
        except Exception:
            transformer_model = None
    if not transformer_available:
        print('SentenceTransformer import succeeded but no pretrained candidate loaded.')
except Exception:
    transformer_available = False

# ---------- Optional XLM-RoBERTa features (Hugging Face) ----------
xlmr_available = False
xlmr_bundle = None
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    xlmr_name = "xlm-roberta-base"
    xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_name)
    xlmr_model = AutoModel.from_pretrained(xlmr_name)
    xlmr_model.eval()
    xlmr_bundle = (xlmr_tokenizer, xlmr_model)
    xlmr_available = True
    print(f"Loaded XLM-R embedder: {xlmr_name}")
except Exception as e:
    print(f"XLM-R unavailable; skipping XLM-R models. Reason: {e}")

def encode_xlmr_texts(texts, bundle, batch_size=16, max_length=192):
    tokenizer, model = bundle
    all_vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            toks = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            out = model(**toks)
            hidden = out.last_hidden_state
            mask = toks["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
            summed = (hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = (summed / counts).cpu().numpy()
            all_vecs.append(pooled)
    return np.vstack(all_vecs) if all_vecs else np.empty((0, 768))

class TransformerEstimator:
    """Simple wrapper that encodes texts with a SentenceTransformer and fits a sklearn classifier."""
    def __init__(self, embedder, clf):
        self.embedder = embedder
        self.clf = clf
    def fit(self, X, y):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = self.embedder.encode(texts, show_progress_bar=False)
        self.clf.fit(embs, y)
        return self
    def predict(self, X):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = self.embedder.encode(texts, show_progress_bar=False)
        return self.clf.predict(embs)

class XLMREstimator:
    """Wrapper that encodes texts with XLM-R and fits a sklearn classifier."""
    def __init__(self, bundle, clf):
        self.bundle = bundle
        self.clf = clf
    def fit(self, X, y):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = encode_xlmr_texts(texts, self.bundle)
        self.clf.fit(embs, y)
        return self
    def predict(self, X):
        texts = [str(t) for t in (X.tolist() if hasattr(X, 'tolist') else list(X))]
        embs = encode_xlmr_texts(texts, self.bundle)
        return self.clf.predict(embs)

# evaluate multiple models including RandomForest and MultinomialNB; tfidf pipelines operate on text, transformer ones on embeddings
models_to_eval = {
    'LinearSVC_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LinearSVC(max_iter=5000))]),
    'LogisticRegression_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=2000, n_jobs=-1))]),
    'MultinomialNB_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', MultinomialNB())]),
    'RandomForest_tfidf': SKPipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced'))]),
}

if transformer_available and transformer_model is not None:
    try:
        models_to_eval['SBERT_Logistic'] = TransformerEstimator(transformer_model, LogisticRegression(max_iter=2000, n_jobs=-1))
        models_to_eval['SBERT_RandomForest'] = TransformerEstimator(transformer_model, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'))
    except Exception as e:
        print(f'Could not instantiate SBERT estimators: {e}')

if xlmr_available and xlmr_bundle is not None:
    try:
        models_to_eval['XLMR_Logistic'] = XLMREstimator(xlmr_bundle, LogisticRegression(max_iter=2000))
    except Exception as e:
        print(f'Could not instantiate XLM-R estimators: {e}')

eval_results = []
for name, m in models_to_eval.items():
    print(f'\nTraining {name}...')
    # Fit: pipelines expect array-like of texts; transformer wrappers accept Series/list
    try:
        if isinstance(m, SKPipeline):
            m.fit(X_tr.values, y_tr.values)
            y_pred = m.predict(X_val.values)
        else:
            m.fit(X_tr, y_tr)
            y_pred = m.predict(X_val)
    except Exception as e:
        print(f'  Error training {name}: {e}')
        continue
    acc = accuracy_score(y_val.values, y_pred)
    bal = balanced_accuracy_score(y_val.values, y_pred)
    prec = precision_score(y_val.values, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_val.values, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val.values, y_pred, average='macro', zero_division=0)
    result_row = {
        'model': name,
        'accuracy': acc,
        'balanced_accuracy': bal,
        'macro_precision': prec,
        'macro_recall': rec,
        'macro_f1': f1,
    }
    eval_results.append(result_row)
    print(f'  Accuracy: {acc:.4f}')
    print(f'  Balanced Accuracy: {bal:.4f}')
    print(f'  Macro Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}')

results_df = pd.DataFrame(eval_results).sort_values('balanced_accuracy', ascending=False)
print('\nModel comparison:')
print(results_df.to_string(index=False))

if results_df.empty:
    raise RuntimeError('No models trained successfully.')

best_model_name = results_df.iloc[0]['model']
print(f'\nBest model for nouns: {best_model_name}')

# Use the best trained estimator to compute validation errors (do this BEFORE refitting on full data)
best_trained = models_to_eval[best_model_name]
if isinstance(best_trained, SKPipeline):
    val_preds = best_trained.predict(X_val.values)
else:
    val_preds = best_trained.predict(X_val)

# Build validation dataframe preserving original indices
val_df = pd.DataFrame({
    'feature_text': X_val.astype(str).values,
    'true_didakta': y_val.astype(str).values,
}, index=X_val.index)
val_df['pred_didakta'] = val_preds
val_df['is_error'] = val_df['true_didakta'] != val_df['pred_didakta']

acc_val = accuracy_score(val_df['true_didakta'], val_df['pred_didakta'])
bal_val = balanced_accuracy_score(val_df['true_didakta'], val_df['pred_didakta'])
print(f'\nValidation accuracy (best pipeline): {acc_val:.4f}, balanced_accuracy: {bal_val:.4f}')

# Extract error rows and merge with original noun_df for full context
errors_idx = val_df.index[val_df['is_error']].tolist()
errors_df = noun_df.loc[errors_idx].copy() if errors_idx else pd.DataFrame()
if not errors_df.empty:
    errors_df = errors_df.assign(
        true_didakta = val_df.loc[errors_idx, 'true_didakta'].values,
        pred_didakta = val_df.loc[errors_idx, 'pred_didakta'].values,
        feature_text = val_df.loc[errors_idx, 'feature_text'].values,
    )

# build context: for each error row extract a window of surrounding rows from noun_df
context_rows = []
window = 2
for idx in errors_idx:
    try:
        pos = noun_df.index.get_loc(idx)
    except KeyError:
        continue
    start = max(0, pos - window)
    end = min(len(noun_df), pos + window + 1)
    slice_df = noun_df.iloc[start:end].copy()
    slice_df['_error_center_index'] = idx
    context_rows.append(slice_df)

context_df = pd.concat(context_rows, ignore_index=False) if context_rows else pd.DataFrame()

# save CSVs with ex1 prefix into results/
errors_path = os.path.join(RESULTS, 'ex1_noun_errors.csv')
context_path = os.path.join(RESULTS, 'ex1_noun_error_context.csv')
errors_df.to_csv(errors_path, index=True, encoding='utf-8-sig')
context_df.to_csv(context_path, index=True, encoding='utf-8-sig')

print(f'\nSaved {len(errors_df)} error rows to: {errors_path}')
print(f'Saved context rows ({len(context_df)}) to: {context_path}')

# Now fit final model on all noun rows for production use
final_pipe = None
best_model_obj = models_to_eval[best_model_name]
if isinstance(best_model_obj, SKPipeline):
    final_pipe = SKPipeline([
        ('tfidf', TfidfVectorizer(max_features=8000, ngram_range=(1,2))),
        ('clf', best_model_obj.named_steps['clf']),
    ])
    final_pipe.fit(X.values, y.values)
else:
    final_pipe = best_model_obj
    try:
        final_pipe.fit(X.values, y.values)
    except Exception:
        final_pipe.fit(X, y)

# expose variables in notebook for inspection
noun_errors_df = errors_df
noun_error_context_df = context_df
print('\nDone. Inspect `noun_errors_df` and `noun_error_context_df`.')

Noun rows kept: 1519
Feature columns (7): ['ref', 'greek', 'lemma', 'pos', 'rel', 'syn', 'line']
Restricted noun task to top 40 labels.
Noun-task rows: 1517
Noun-task unique labels: 40
XLM-R unavailable; skipping XLM-R models. Reason: No module named 'torch'

Training LinearSVC_tfidf...


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Accuracy: 0.7796
  Balanced Accuracy: 0.4264
  Macro Precision: 0.4346, Recall: 0.3980, F1: 0.4034

Training LogisticRegression_tfidf...
  Accuracy: 0.6908
  Balanced Accuracy: 0.2072
  Macro Precision: 0.2005, Recall: 0.2072, F1: 0.1874

Training MultinomialNB_tfidf...
  Accuracy: 0.5592
  Balanced Accuracy: 0.0829
  Macro Precision: 0.0759, Recall: 0.0829, F1: 0.0688

Training RandomForest_tfidf...


c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\Farnoosh\Documents\GitHub\Didakta\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Accuracy: 0.7467
  Balanced Accuracy: 0.3804
  Macro Precision: 0.4056, Recall: 0.3551, F1: 0.3568

Model comparison:
                   model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1
         LinearSVC_tfidf  0.779605           0.426418         0.434613      0.397990  0.403448
      RandomForest_tfidf  0.746711           0.380425         0.405579      0.355063  0.356830
LogisticRegression_tfidf  0.690789           0.207162         0.200484      0.207162  0.187383
     MultinomialNB_tfidf  0.559211           0.082945         0.075868      0.082945  0.068764

Best model for nouns: LinearSVC_tfidf

Validation accuracy (best pipeline): 0.7796, balanced_accuracy: 0.4264

Saved 67 error rows to: c:\\Users\Farnoosh\Documents\GitHub\Didakta\ex1_noun_errors.csv
Saved context rows (335) to: c:\\Users\Farnoosh\Documents\GitHub\Didakta\ex1_noun_error_context.csv

Done. Inspect `noun_errors_df` and `noun_error_context_df`.


In [31]:
# Optional transformer-only experiment in a separate cell (XLM-RoBERTa)
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

if not all(v in globals() for v in ["X_tr", "X_val", "y_tr", "y_val"]):
    raise RuntimeError("Run Cell 1 first so train/validation splits exist.")

try:
    import torch
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    xlmr_eval_df = pd.DataFrame()
    print(f"Skipping XLM-R cell: missing dependency ({e}). Install `torch` and `transformers`.")
else:
    xlmr_name = "xlm-roberta-base"
    print(f"Loading {xlmr_name}...")
    tokenizer = AutoTokenizer.from_pretrained(xlmr_name)
    model = AutoModel.from_pretrained(xlmr_name)
    model.eval()

    def encode_xlmr_texts(texts, batch_size=16, max_length=192):
        vectors = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i + batch_size]
                toks = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt",
                )
                out = model(**toks)
                hidden = out.last_hidden_state
                mask = toks["attention_mask"].unsqueeze(-1).expand(hidden.size()).float()
                summed = (hidden * mask).sum(dim=1)
                counts = mask.sum(dim=1).clamp(min=1e-9)
                pooled = (summed / counts).cpu().numpy()
                vectors.append(pooled)
        return np.vstack(vectors) if vectors else np.empty((0, 768))

    Xtr_text = [str(t) for t in (X_tr.tolist() if hasattr(X_tr, "tolist") else list(X_tr))]
    Xval_text = [str(t) for t in (X_val.tolist() if hasattr(X_val, "tolist") else list(X_val))]

    print("Encoding train texts with XLM-R...")
    Xtr_emb = encode_xlmr_texts(Xtr_text)
    print("Encoding validation texts with XLM-R...")
    Xval_emb = encode_xlmr_texts(Xval_text)

    xlmr_clf = LogisticRegression(max_iter=2000)
    xlmr_clf.fit(Xtr_emb, y_tr.values)
    xlmr_pred = xlmr_clf.predict(Xval_emb)

    xlmr_eval = {
        "model": "XLMR_Logistic",
        "accuracy": accuracy_score(y_val.values, xlmr_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val.values, xlmr_pred),
        "macro_precision": precision_score(y_val.values, xlmr_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_val.values, xlmr_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_val.values, xlmr_pred, average="macro", zero_division=0),
    }
    xlmr_eval_df = pd.DataFrame([xlmr_eval])
    print(xlmr_eval_df.to_string(index=False))

Skipping XLM-R cell: missing dependency (No module named 'torch'). Install `torch` and `transformers`.
